# API-232 Lab: Building a Fantasy Football Team Under a Salary Cap
### Linear programming, binding constraints, the optimizer's curse and matching, with real NFL data

In **fantasy football**, you pick real NFL players for your team and score points based on what they do on the field each week: yards, catches and touchdowns. In this lab you'll act as a fantasy general manager with a twist: you must pay each player his **real NFL salary**, and your total payroll can't exceed a cap.

That turns team-building into the same kind of problem as Quiz 3's vaccination drive: **maximize an objective subject to resource constraints.** Here the objective is projected fantasy points, and the constraints are a salary cap and roster rules.

Then we'll ask two questions that matter as much in policy as in fantasy football:

* **Does the "optimal" plan deliver what it promised?** When you pick the options with the best forecasts, you also pick the ones whose forecasts were most likely too high.
* **How should you assign scarce resources when several people want the same thing?** Fantasy leagues assign newly popular players through a "waiver wire", a small matching market like the summer-jobs question on the quiz.

**Concepts from Quiz 3:**

| Part | Concept | Quiz link |
|---|---|---|
| 1 | Objective, constraints, feasibility, binding constraints | Question 2(a)–(c) |
| 2 | Solving the problem; the value of relaxing a constraint | Question 2(c) |
| 3 | Can adding a rule ever help? | Question 2(d) |
| 4 | The optimizer's curse: forecasts of the chosen plan are too optimistic | New: prediction meets optimization (Question 1) |
| 5 | Boston mechanism vs deferred acceptance, stability, strategy | Question 3 |

**Time:** about 75 minutes. Answer the written questions in the markdown cells marked **✏️ Your answer**.


## 0. Setup (about 3 minutes)

Run the next cell. It installs `nflreadpy`, the Python package for the free [nflverse](https://github.com/nflverse) NFL data, and downloads two tables:

* **Season statistics** (2013–2025) for every quarterback, running back, wide receiver and tight end, including fantasy points.
* **Contracts:** every NFL contract since about 2011, from OverTheCap, including the average salary per year (`apy`, in millions of dollars).

It takes about a minute.

In [ ]:
!pip install -q nflreadpy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from scipy.optimize import milp, linprog, LinearConstraint, Bounds
import nflreadpy as nfl

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 20)

contracts = nfl.load_contracts().to_pandas()
stats = nfl.load_player_stats(list(range(2013, 2026)), summary_level="reg").to_pandas()
stats = stats[stats.position.isin(["QB", "RB", "WR", "TE"])]
print(f"{len(contracts):,} contracts, {len(stats):,} player-seasons")

### The player pool (provided)

`build_pool(season)` returns every player we can pick for a season, with three numbers:

* **`salary`**: his real NFL salary that season, in millions of dollars (the average per year of the contract in force).
* **`proj`**: his **projected** fantasy points, made the simplest way: last season's points per game × 17 games. This is the forecast you have when you build your team before the season.
* **`actual`**: the fantasy points he **actually** scored that season (0 if he didn't play). You don't get to see this until Part 4.

Points use standard "PPR" scoring (a point per catch, a point per 10 yards, 6 per touchdown). Only players who played at least 6 games last season are included.

In [ ]:
def build_pool(season):
    prev = stats[(stats.season == season - 1) & (stats.games >= 6)]
    pool = prev[["player_id", "player_display_name", "position", "recent_team", "games", "fantasy_points_ppr"]].rename(
        columns={"player_display_name": "player", "recent_team": "team_prev",
                 "games": "games_prev", "fantasy_points_ppr": "points_prev"})
    pool["proj"] = (pool.points_prev / pool.games_prev * 17).round(1)

    # the contract in force during `season`
    c = contracts[contracts.gsis_id.notna() & (contracts.apy > 0) & (contracts.year_signed <= season)
                  & (contracts.year_signed + contracts.years.fillna(1) > season)]
    c = c.sort_values("year_signed").groupby("gsis_id").tail(1)
    pool = pool.merge(c[["gsis_id", "apy"]], left_on="player_id", right_on="gsis_id").rename(columns={"apy": "salary"})
    pool["salary"] = pool.salary.round(2)

    this = stats[stats.season == season].set_index("player_id")
    pool["team"] = pool.player_id.map(this.recent_team).fillna(pool.team_prev)
    pool["actual"] = pool.player_id.map(this.fantasy_points_ppr).fillna(0).round(1)
    pool["season"] = season
    return (pool[["season", "player", "position", "team", "salary", "games_prev", "proj", "actual"]]
            .sort_values("proj", ascending=False).reset_index(drop=True))

pool = build_pool(2025)
print(len(pool), "players")
pool.head(10)

## Part 1. Writing down the problem (about 12 minutes)

**The rules.** Your team has exactly **8 players**:

| Slot | Requirement |
|---|---|
| Quarterback (QB) | exactly 1 |
| Running back (RB) | 2 or 3 |
| Wide receiver (WR) | 3 or 4 |
| Tight end (TE) | 1 or 2 |
| Total | exactly 8 (the 8th player is a "flex": an extra RB, WR or TE) |

Your total salary can't exceed **$50 million**. (For scale, a single star quarterback earns about $50 million a year.) Your goal is the most projected fantasy points.

**1a.** Let $x_i = 1$ if you pick player $i$ and $0$ otherwise. Let $p_i$ be his projected points and $s_i$ his salary. Write the objective and the constraints. You can use sums like $\sum_{i \in QB} x_i$.

✏️ **Your answer:**


**1b.** Four friends propose teams. For each one, compute the total salary, projected points and the number of players at each position. Then decide whether it is feasible.

In [ ]:
proposals = {
    "A: last year's stars": ["Lamar Jackson", "Saquon Barkley", "Jahmyr Gibbs", "Bijan Robinson",
                             "Ja'Marr Chase", "Chris Godwin Jr.", "Puka Nacua", "George Kittle"],
    "B: rookie contracts":  ["Bo Nix", "Jahmyr Gibbs", "De'Von Achane", "Kenneth Walker III",
                             "Puka Nacua", "Brian Thomas Jr.", "Ladd McConkey", "Brock Bowers"],
    "C: two quarterbacks":  ["Jayden Daniels", "Bo Nix", "Jahmyr Gibbs", "Bijan Robinson",
                             "Puka Nacua", "Malik Nabers", "Brian Thomas Jr.", "Brock Bowers"],
    "D: best points per $": ["Aidan O'Connell", "Ameer Abdullah", "Audric Estimé", "Puka Nacua",
                             "Ray-Ray McCloud", "Kevin Austin Jr.", "Jordan Akins", "Josh Whyle"],
}

def summarize(names, pool=pool):
    team = pool[pool.player.isin(names)]
    # TODO: return a dict with total salary, total projected points, and the count of each position
    ...

pd.DataFrame({name: summarize(names) for name, names in proposals.items()}).T

**1c.** Which proposals are feasible? Among the feasible ones, which projects the most points? For that team, which constraints are **binding** (used up exactly)?

✏️ **Your answer:**

**1d.** Proposal D picks the players with the most projected points per dollar of salary. Why doesn't that produce the best team?

✏️ **Your answer:**


## Part 2. Solving it (about 15 minutes)

Checking proposals by hand doesn't scale: there are hundreds of millions of possible teams. A **solver** searches them efficiently. `scipy.optimize.milp` solves problems of the form:

> minimize $c \cdot x$ subject to $\text{lower} \le A x \le \text{upper}$, with chosen $x$ restricted to integers

Each **row** of $A$ is one constraint: it gives, for each player, how much of that resource he uses. `milp` *minimizes*, so to maximize points we minimize **negative** points.

**2a.** Fill in the constraint rows below. The salary row is done for you.

In [ ]:
SLOTS = {"QB": (1, 1), "RB": (2, 3), "WR": (3, 4), "TE": (1, 2)}   # (min, max) of each position

def solve_lineup(pool, cap, score="proj", extra_rules=(), relax=False):
    """Returns x: a 0/1 (or fractional, if relax=True) choice for each player.
    extra_rules: a list of (row, lower, upper) tuples added as constraints (used in Part 3)."""
    n = len(pool)
    rows, lo, hi = [pool.salary.values], [0], [cap]          # salary: 0 <= total salary <= cap

    # TODO: add one row per position (1.0 for players at that position, 0.0 otherwise) with its min and max
    ...
    # TODO: add a row that counts all players, with lower = upper = 8
    ...

    assert len(rows) == 6, "Add the four position rows and the roster-size row before solving"
    for r, a, b in extra_rules:
        rows.append(np.asarray(r, float)); lo.append(a); hi.append(b)
    res = milp(c=-pool[score].values, constraints=LinearConstraint(np.vstack(rows), lo, hi),
               integrality=np.zeros(n) if relax else np.ones(n), bounds=Bounds(0, 1))
    return res.x

CAP = 50
x = solve_lineup(pool, CAP)
best = pool[x > 0.5]
print(f"Salary ${best.salary.sum():.2f}M, projected {best.proj.sum():.1f} points")
best

**2b.** How does the optimal team compare with proposal B? What kind of contract do most of its players have, and why does that make sense given how NFL salaries are set? (Hint: look at how long each player has been in the league; players drafted in the last few years are paid a fixed rookie scale.)

✏️ **Your answer:**


**2c. How much is more cap space worth?** Solve the problem for caps from $5 million to $150 million and plot the best projected points against the cap.

In [ ]:
caps = [5, 10, 15, 20, 25, 30, 40, 50, 60, 80, 100, 125, 150]
# TODO: for each cap, solve and record the best projected points
value = ...

plt.plot(caps, value, marker="o")
plt.xlabel("Salary cap ($ millions)"); plt.ylabel("Best projected points")
plt.title("Each extra dollar of cap buys fewer points"); plt.grid(alpha=0.3); plt.show()

**2d. Fractions of players.** On the quiz, fractional station-hours were allowed. What happens if we allow fractional players? Run the **relaxed** problem, where each $x_i$ can be anything between 0 and 1.

In [ ]:
xr = solve_lineup(pool, CAP, relax=True)
relaxed = pool.assign(x=xr.round(3))[xr > 1e-6]
print(f"Relaxed optimum: {(xr * pool.proj).sum():.1f} points, salary ${(xr * pool.salary).sum():.2f}M")
relaxed[["player", "position", "salary", "proj", "x"]]

In a relaxed (fractional) problem, the solver also reports a **shadow price** for each constraint: how much the objective would rise if that constraint were loosened by one unit. The cell below gets the shadow price of the salary cap.

In [ ]:
n = len(pool)
P = lambda pos: (pool.position == pos).values.astype(float)
A_ub = np.vstack([pool.salary.values, P("RB"), P("WR"), P("TE"), -P("RB"), -P("WR"), -P("TE")])
b_ub = [CAP, 3, 4, 2, -2, -3, -1]
A_eq = np.vstack([P("QB"), np.ones(n)])
lp = linprog(-pool.proj.values, A_ub=A_ub, b_ub=b_ub, A_eq=A_eq, b_eq=[1, 8], bounds=(0, 1), method="highs")
shadow = -lp.ineqlin.marginals[0]
print(f"Shadow price of the cap: {shadow:.2f} projected points per extra $1 million")

x51 = solve_lineup(pool, 51)
print(f"Integer optimum at $50M: {best.proj.sum():.1f}   at $51M: {pool[x51 > 0.5].proj.sum():.1f}")

**2e.** Answer the following:

1. In the relaxed solution, what fraction of which players does the solver pick? Why can't you just round it?
2. At the integer optimum, the team spends only $48.15 million, so the $50 million cap has slack. Is the cap binding? Does raising it to $51 million help? What does that tell you about "binding" in integer problems?
3. Compare the shadow price with what one extra $1 million actually buys in the integer problem.

✏️ **Your answer:**


## Part 3. Adding rules (about 10 minutes)

On the quiz, the county added a rule: at least 50 station-hours must be mobile vans. Here, a league commissioner is considering two rules.

* **Spread the talent:** no more than one player from any NFL team.
* **Veteran minimum:** at least two players earning $10 million or more, so fantasy teams look like "real" NFL rosters.

**3a.** Before you run anything: can either rule *increase* the best projected points? Can either one leave it unchanged?

✏️ **Your answer:**


**3b.** Add each rule with `extra_rules` and compare.

In [ ]:
base = best.proj.sum()

# Rule 1: at most one player per NFL team (one row per team)
spread = [((pool.team == t).values.astype(float), 0, 1) for t in pool.team.unique()]
# TODO: solve with extra_rules=spread and report the best projected points and the cost relative to `base`
...

# Rule 2: at least two players with salary >= 10
veteran = ...   # TODO: a list with one (row, lower, upper) tuple
...

**3c.** Which rule is binding? How many projected points does the veteran rule cost, and how much is that per dollar of the cap's shadow price from 2e? Suppose the commissioner says the veteran rule makes the game "fairer" to established players. How would you present the trade-off to the league?

✏️ **Your answer:**


## Part 4. Did the plan deliver? The optimizer's curse (about 18 minutes)

So far we've trusted the projections. Now look at what the players **actually** scored in 2025.

**4a.** Compare your optimal team's projected and actual points.

In [ ]:
# TODO: show each player's projected and actual points, and the team totals
...

**Is that just bad luck?** The optimizer picked the players whose projections were *highest relative to their salary*. But a projection is the truth plus an error. Players whose projections are unusually high are, on average, players whose errors happened to be positive. So the team you pick tends to be the one whose forecast was most over-optimistic, even if each individual forecast is unbiased. This is called the **optimizer's curse**.

**4b. A simulation.** Pretend there are 400 players. Each has a true value, and our projection = true value + random noise with **mean zero**, so every individual projection is unbiased.

In [ ]:
rng = np.random.default_rng(232)
over_best, over_random = [], []
for _ in range(2000):
    truth = rng.normal(200, 50, 400)
    proj = truth + rng.normal(0, 60, 400)          # unbiased: errors average zero
    # TODO: pick the 8 players with the highest projections; record (their projected total - their true total)
    ...
    # TODO: pick 8 players at random; record the same
    ...

print(f"Average overestimate, top-8 by projection: {np.mean(over_best):.0f} points")
print(f"Average overestimate, random 8:            {np.mean(over_random):.0f} points")

**4c. Calibrating the forecast.** A fix is to ask: historically, when a player was projected for $p$ points, how many did he actually score? Fit that relationship on **earlier seasons only**, then use it to forecast. The cell below does this for every season from 2016 to 2025. For each season it reports the optimal team's naive projection, its **calibrated** forecast, its actual points, and whether optimizing on the calibrated forecast would have picked a different team.

In [ ]:
pools = {s: build_pool(s) for s in range(2014, 2026)}
rows = []
for s in range(2016, 2026):
    train = pd.concat([pools[t] for t in range(2014, s)])
    calib = smf.ols("actual ~ proj * C(position)", train).fit()   # actual points as a function of projection
    p = pools[s].copy()
    p["calibrated"] = calib.predict(p)
    naive = p[solve_lineup(p, CAP) > 0.5]
    recal = p[solve_lineup(p, CAP, score="calibrated") > 0.5]
    rows.append({"season": s, "projected": naive.proj.sum(), "calibrated": naive.calibrated.sum(),
                 "actual": naive.actual.sum(), "same team?": set(naive.player) == set(recal.player),
                 "actual, calibrated team": recal.actual.sum()})
curse = pd.DataFrame(rows).set_index("season").round(0)
print(curse)
print("\nAverages:"); print(curse.drop(columns="same team?").mean().round(0))

**4d.** Answer the following:

1. In the simulation, every projection was unbiased. Why does the top-8 team still disappoint, while the random team doesn't?
2. In the real data, how far off was the naive projection on average, and how close was the calibrated forecast?
3. Calibrating changed the forecast a lot. Did it usually change the team? Explain why shrinking every projection toward the average by a similar factor tends to leave the ranking, and hence the choice, unchanged.

✏️ **Your answer:**

**4e. Policy.** A foundation funds pilot programs and scales up the three with the largest estimated effects. When it scales them up, the effects are consistently smaller than in the pilots. Using this part of the lab, explain why that should be expected even if every pilot evaluation was done correctly, and what the foundation should do about it.

✏️ **Your answer:**


## Part 5. The waiver wire as a matching market (about 12 minutes)

During the season, a player who wasn't on anyone's fantasy team can suddenly break out. When several managers want him, the league uses the **waiver wire**: each manager submits a ranked list of claims, and the league has a **priority order**. Usually the worst-placed team has the highest priority, and the same order applies to every player.

After week 5 of 2025, three unclaimed players had just had big games:

In [ ]:
wk = nfl.load_player_stats([2025], summary_level="week").to_pandas()
wk = wk[wk.season_type == "REG"]
names = ["Rico Dowdle", "Kendrick Bourne", "AJ Barner"]
show = wk[wk.player_display_name.isin(names)]
summary = show.groupby(["player_display_name", "position"]).apply(lambda d: pd.Series({
    "avg weeks 1-4": d[d.week < 5].fantasy_points_ppr.mean(),
    "week 5": d[d.week == 5].fantasy_points_ppr.sum()}), include_groups=False).round(1)
summary

Three managers want them. Their rankings (from best to worst) and the waiver priority:

| Manager | 1st choice | 2nd choice | 3rd choice |
|---|---|---|---|
| Ana (priority 1) | Dowdle | Barner | Bourne |
| Ben (priority 2) | Dowdle | Bourne | Barner |
| Cal (priority 3) | Bourne | Dowdle | Barner |

Every player has the same priority over managers: **Ana > Ben > Cal**. The functions below implement both mechanisms from Quiz 3.

In [ ]:
prefs = {"Ana": ["Dowdle", "Barner", "Bourne"],
         "Ben": ["Dowdle", "Bourne", "Barner"],
         "Cal": ["Bourne", "Dowdle", "Barner"]}
priority = ["Ana", "Ben", "Cal"]                     # same order for every player
players = ["Dowdle", "Bourne", "Barner"]

def boston(prefs, priority):
    """Round k: each unassigned manager claims his k-th choice; each player goes to the
    highest-priority claimant if still available. Assignments are final."""
    match, taken = {}, set()
    for k in range(len(players)):
        claims = {}
        for m, ranking in prefs.items():
            if m not in match and ranking[k] not in taken:
                claims.setdefault(ranking[k], []).append(m)
        for p, ms in claims.items():
            winner = min(ms, key=priority.index)
            match[winner] = p; taken.add(p)
    return match

def deferred_acceptance(prefs, priority):
    """Managers apply in order of preference; each player tentatively holds the
    highest-priority applicant so far and rejects the rest, until no one is rejected."""
    nxt = {m: 0 for m in prefs}; held = {}
    free = list(prefs)
    while free:
        m = free.pop(0); p = prefs[m][nxt[m]]; nxt[m] += 1
        if p not in held:
            held[p] = m
        elif priority.index(m) < priority.index(held[p]):
            free.append(held[p]); held[p] = m
        else:
            free.append(m)
    return {m: p for p, m in held.items()}

def justified_envy(match, prefs, priority):
    """Pairs (manager, player) where the manager prefers the player to his own
    and has higher priority than the manager who got him."""
    owner = {p: m for m, p in match.items()}
    out = []
    for m, ranking in prefs.items():
        for p in ranking[:ranking.index(match[m])]:
            if priority.index(m) < priority.index(owner[p]):
                out.append((m, p))
    return out

**5a.** Run both mechanisms. Report each assignment and any justified envy.

In [ ]:
# TODO: run boston() and deferred_acceptance(), and check each result with justified_envy()
...

**5b.** Under the Boston mechanism, can Ben get a player he truly prefers by submitting a different ranking? Try it in code.

In [ ]:
# TODO: change Ben's submitted ranking and rerun boston()
...

**5c.** Answer the following:

1. Which mechanism is stable? Who has justified envy under the other one?
2. Why does Ben benefit from *not* listing the player he wants most? Would he benefit from that under deferred acceptance?
3. When every player has the same priority order, deferred acceptance gives the same result as letting managers pick one at a time in priority order (a "serial dictatorship"). Why? (This is what most fantasy leagues actually do.)
4. The commissioner says the league should use Boston because it gives more managers their first choice. Is that a good argument?

✏️ **Your answer:**


**5d. What happened next?** Run the cell below to see how each player scored for the rest of the season.

In [ ]:
after = show[show.week > 5].groupby("player_display_name").fantasy_points_ppr.mean().round(1)
pd.DataFrame({"week 5": show[show.week == 5].set_index("player_display_name").fantasy_points_ppr,
              "avg after week 5": after})

Connect this to Part 4: what does it suggest about how much any manager should bid for last week's breakout player?

✏️ **Your answer:**


## Part 6. Bringing it together (about 5 minutes)

**6.** A city housing agency uses an optimization model to choose which of 200 proposed affordable-housing projects to fund under a fixed budget, based on each project's estimated number of units and cost. A council member wants to add a rule that at least 30% of the funded projects must be in the outer boroughs. In four or five sentences, use what you learned in this lab to advise the agency. Address what the rule can and can't do to the model's objective, how to report its cost, and how much to trust the model's estimate of total units delivered.

✏️ **Your answer:**
